# 🛠️ Chapter 02-01b: Hands-On — Prompt Patterns That Work

---

## Time to Master the Art of Talking to LLMs! 🎨

In the concept notebook, we learned the theory of prompt engineering. Now let's **practice every pattern** until it becomes second nature.

By the end of this notebook, you'll have:
- ✅ Practiced zero-shot, one-shot, and few-shot prompting
- ✅ Built prompts with role assignment and personas
- ✅ Used LangChain's PromptTemplates like a pro
- ✅ Extracted structured outputs (JSON, tables)
- ✅ Built a real product review analyzer

Let's go! 🚀

---

## 📦 Setup

In [ ]:
# !pip install openai langchain-openai langchain-core python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

# We'll use this model throughout
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

api_key = os.getenv("OPENAI_API_KEY")
print(f"✅ Ready!" if api_key else "❌ Set OPENAI_API_KEY in .env")

---

## 🧪 Exercise 1: Zero-Shot, One-Shot, Few-Shot

These are the **foundational prompting patterns**. The names describe how many EXAMPLES you give:

```
╔═══════════════════════════════════════════════════════════════════════╗
║                    SHOT TERMINOLOGY                                   ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  🎯 ZERO-SHOT:  No examples. Just the instruction.                   ║
║     "Classify this review as positive or negative: ..."              ║
║                                                                       ║
║  1️⃣ ONE-SHOT:   One example showing the desired format.              ║
║     "Example: 'Great product!' → positive"                           ║
║     "Now classify: ..."                                              ║
║                                                                       ║
║  📚 FEW-SHOT:   Multiple examples (2-5 usually).                     ║
║     "Example 1: ... → positive"                                      ║
║     "Example 2: ... → negative"                                      ║
║     "Example 3: ... → neutral"                                       ║
║     "Now classify: ..."                                              ║
║                                                                       ║
║  💡 More examples = better consistency but more tokens (cost!)       ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

In [ ]:
# ZERO-SHOT: No examples, just instructions
review = "The battery life on this phone is terrible, but the camera is amazing."

zero_shot_prompt = f"""Classify this product review as 'positive', 'negative', or 'mixed'.

Review: \"{review}\"

Classification:"""

response = llm.invoke(zero_shot_prompt)
print("🎯 ZERO-SHOT Result:")
print(f"   Review: \"{review}\"")
print(f"   Classification: {response.content}")

In [ ]:
# FEW-SHOT: Give examples to show the exact format you want
few_shot_prompt = f"""Classify product reviews as 'positive', 'negative', or 'mixed'.
Respond with ONLY the classification word.

Examples:
Review: "Absolutely love this laptop! Fast, lightweight, perfect."
Classification: positive

Review: "Broke after 2 days. Worst purchase ever. Total waste of money."
Classification: negative

Review: "Good sound quality but uncomfortable to wear for long periods."
Classification: mixed

Now classify:
Review: \"{review}\"
Classification:"""

response = llm.invoke(few_shot_prompt)
print("📚 FEW-SHOT Result:")
print(f"   Review: \"{review}\"")
print(f"   Classification: {response.content}")

### 🔍 Notice the Difference?

```
┌─────────────────────────────────────────────────────────────────────┐
│  Zero-shot: Model might give verbose answer or inconsistent format │
│  Few-shot:  Model follows the EXACT format from examples!          │
│                                                                     │
│  💡 Rule: When you need CONSISTENT output format, use few-shot.    │
│     When the task is simple/obvious, zero-shot is fine.            │
└─────────────────────────────────────────────────────────────────────┘
```

---

## 🧪 Exercise 2: Role Prompting (Persona Assignment)

The system prompt is where you assign a **role** or **persona** to the LLM. This drastically changes its behavior.

In [ ]:
# Different roles produce VERY different answers

question = "Should I use Python or JavaScript for my next project?"

roles = {
    "🐍 Python Dev": "You are a passionate Python developer with 10 years of experience. You strongly prefer Python.",
    "🌐 JS Dev": "You are a passionate JavaScript developer with 10 years of experience. You strongly prefer JavaScript.",
    "⚖️ Neutral CTO": "You are a neutral CTO who evaluates technologies objectively. Give a balanced analysis.",
}

for role_name, system_prompt in roles.items():
    response = llm.invoke([
        SystemMessage(content=system_prompt + " Keep your answer to 2-3 sentences."),
        HumanMessage(content=question)
    ])
    print(f"\n{role_name}:")
    print(f"  {response.content}")
    print("-" * 70)

### 💡 Key Insight:

> Same question, 3 completely different answers. The system prompt makes the LLM **adopt a perspective**. This is incredibly powerful for building specialized AI assistants — a medical AI, a legal AI, a coding tutor, etc.

---

## 🧪 Exercise 3: LangChain PromptTemplates

In real projects, you don't hardcode prompts as strings. You use **templates** — reusable prompts with fillable variables.

```
╔═══════════════════════════════════════════════════════════════════════╗
║              WHY TEMPLATES?                                           ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  Hardcoded:  f"Translate {text} to {language}"  ← messy at scale    ║
║                                                                       ║
║  Template:   ChatPromptTemplate with variables   ← clean, reusable  ║
║              • Version controlled                                    ║
║              • Composable with chains                                ║
║              • Type-safe                                              ║
║              • Shareable across team                                  ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Create a reusable prompt template
translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a professional translator. Translate accurately and naturally."),
    ("human", "Translate the following {source_language} text to {target_language}:\n\n{text}")
])

# Use the template with different inputs!
tests = [
    {"source_language": "English", "target_language": "Hindi", "text": "Hello, how are you?"},
    {"source_language": "English", "target_language": "French", "text": "I love programming"},
    {"source_language": "English", "target_language": "Japanese", "text": "Where is the train station?"},
]

for test in tests:
    # Format the template with variables
    messages = translate_prompt.format_messages(**test)
    response = llm.invoke(messages)
    
    print(f"🌐 {test['source_language']} → {test['target_language']}")
    print(f"   Input:  {test['text']}")
    print(f"   Output: {response.content}")
    print()

### 🔗 Chaining Template + Model (LCEL Preview!)

This is a sneak peek at **LCEL (LangChain Expression Language)** — the `|` pipe operator chains steps together:

In [ ]:
# The LCEL way — template | model = chain!
# This is how real LangChain apps are built

translate_chain = translate_prompt | llm

# Now invoke the CHAIN (not the model directly)
result = translate_chain.invoke({
    "source_language": "English",
    "target_language": "Spanish",
    "text": "Generative AI is transforming how we build software."
})

print(f"🔗 Chain result: {result.content}")
print(f"\n💡 This is LCEL! prompt | model = chain")
print(f"   We'll master this in Chapter 3.")

---

## 🧪 Exercise 4: Structured Output (JSON)

In production, you almost always need **structured output** — JSON that your code can parse. Not free-form text.

```
╔═══════════════════════════════════════════════════════════════════════╗
║           THE #1 PRODUCTION PROMPTING PATTERN                         ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  ❌ "Tell me about this product" → free text (hard to parse)         ║
║                                                                       ║
║  ✅ "Analyze this product and return JSON with:                      ║
║      - sentiment (positive/negative/mixed)                           ║
║      - confidence (0.0-1.0)                                           ║
║      - key_points (list of strings)"                                 ║
║     → structured, parseable, reliable                                ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

In [ ]:
import json

# Template for structured analysis
analysis_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a product review analyst. 
Analyze reviews and return ONLY valid JSON with this exact format:
{{
    "sentiment": "positive" | "negative" | "mixed",
    "confidence": 0.0-1.0,
    "key_points": ["point1", "point2"],
    "summary": "one sentence summary"
}}
Return ONLY the JSON, no other text."""),
    ("human", "Analyze this review:\n\n{review}")
])

# Build the chain
analysis_chain = analysis_prompt | llm

# Test reviews
reviews = [
    "Absolutely love this laptop! The screen is gorgeous, battery lasts all day, and it's super fast. Best purchase I've made this year!",
    "The headphones sound great but they're really uncomfortable after 30 minutes. Also, the Bluetooth keeps disconnecting.",
    "Complete waste of money. Broke after 3 days. Customer support was useless. DO NOT BUY.",
]

print("📊 PRODUCT REVIEW ANALYZER")
print("=" * 60)

for review in reviews:
    result = analysis_chain.invoke({"review": review})
    
    # Parse the JSON
    try:
        analysis = json.loads(result.content)
        print(f"\n📝 Review: \"{review[:60]}...\"")
        print(f"   Sentiment:   {analysis['sentiment']}")
        print(f"   Confidence:  {analysis['confidence']}")
        print(f"   Key Points:  {analysis['key_points']}")
        print(f"   Summary:     {analysis['summary']}")
    except json.JSONDecodeError:
        print(f"   ⚠️ Failed to parse JSON: {result.content[:100]}")

### 💡 Key Insight:

> We can now process 1000s of reviews programmatically! The model returns structured JSON that our code can parse, aggregate, and use. This is **how real GenAI products work** — LLMs as a data processing layer.

---

## 🧪 Exercise 5: Output Formatting Patterns

Different use cases need different output formats. Let's practice:

In [ ]:
# Pattern: Get a TABLE output

table_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a tech comparison expert. Always respond with a markdown table."),
    ("human", "Compare {items} across these dimensions: {dimensions}")
])

chain = table_prompt | llm
result = chain.invoke({
    "items": "Python, JavaScript, and Go",
    "dimensions": "learning curve, performance, web development, data science, job market"
})

print(result.content)

In [ ]:
# Pattern: Get a STEP-BY-STEP output

steps_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a coding tutor. Always respond with:
1. A brief explanation of the concept
2. Step-by-step approach (numbered)
3. Python code example
4. Common pitfalls to avoid"""),
    ("human", "{question}")
])

chain = steps_prompt | llm
result = chain.invoke({"question": "How do I reverse a linked list?"})

print(result.content)

---

## 🚀 Build: Product Review Analyzer Pipeline

Let's combine everything into a real mini-project!

In [ ]:
import json

# Full review analysis pipeline

analyzer_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a product review analysis engine.
Analyze the review and return ONLY valid JSON:
{{
    "sentiment": "positive" | "negative" | "mixed",
    "score": 1-5,
    "emotions": ["list of detected emotions"],
    "pros": ["list of positive points"],
    "cons": ["list of negative points"],
    "would_recommend": true | false,
    "one_line_summary": "single sentence summary"
}}"""),
    ("human", "Review: {review}")
])

analyzer_chain = analyzer_prompt | llm

# Batch of reviews to analyze
batch_reviews = [
    "This phone is incredible! Camera quality rivals professional cameras, the battery easily lasts 2 days, and it's surprisingly affordable. My only complaint is the slightly heavy weight.",
    "Ordered this 3 weeks ago and it arrived damaged. Tried returning it but customer service put me on hold for 45 minutes. The product itself looks cheap and flimsy. Save your money.",
    "Decent laptop for the price. Nothing special but gets the job done for basic tasks. Wouldn't use it for gaming or video editing though. Good for students on a budget.",
]

print("📊 BATCH REVIEW ANALYSIS")
print("=" * 60)

results = []
for i, review in enumerate(batch_reviews, 1):
    result = analyzer_chain.invoke({"review": review})
    try:
        analysis = json.loads(result.content)
        results.append(analysis)
        
        print(f"\n{'─' * 60}")
        print(f"📝 Review #{i}: \"{review[:50]}...\"")
        print(f"   Sentiment: {analysis['sentiment']} (Score: {analysis['score']}/5)")
        print(f"   Emotions:  {', '.join(analysis['emotions'])}")
        print(f"   Pros:      {analysis.get('pros', [])}")
        print(f"   Cons:      {analysis.get('cons', [])}")
        print(f"   Recommend: {'✅ Yes' if analysis['would_recommend'] else '❌ No'}")
        print(f"   Summary:   {analysis['one_line_summary']}")
    except json.JSONDecodeError:
        print(f"   ⚠️ Parse error")

# Summary stats
if results:
    avg_score = sum(r['score'] for r in results) / len(results)
    recommend_pct = sum(1 for r in results if r['would_recommend']) / len(results) * 100
    print(f"\n{'═' * 60}")
    print(f"📈 SUMMARY: Avg Score: {avg_score:.1f}/5 | Recommend Rate: {recommend_pct:.0f}%")

---

## ✍️ Practice: Build a Code Reviewer

**Challenge:** Create a prompt template that reviews Python code and returns structured feedback.

In [ ]:
# YOUR CODE HERE

# TODO: Create a ChatPromptTemplate for code review
#   System: "You are a senior Python developer reviewing code..."
#   Human: takes {code} variable
#   Output format: JSON with fields like quality_score, issues, suggestions, improved_code

# TODO: Create the chain: prompt | llm

# TODO: Test with this code:
test_code = """
def find_max(lst):
    max = lst[0]
    for i in range(len(lst)):
        if lst[i] > max:
            max = lst[i]
    return max
"""

# result = code_review_chain.invoke({"code": test_code})
# print(result.content)

<details>
<summary>💡 Click to see solution</summary>

```python
code_review_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a senior Python developer. Review the code and return JSON:
{{
    "quality_score": 1-10,
    "issues": ["list of problems found"],
    "suggestions": ["list of improvements"],
    "improved_code": "the corrected code"
}}"""),
    ("human", "Review this Python code:\n```python\n{code}\n```")
])

code_review_chain = code_review_prompt | llm
result = code_review_chain.invoke({"code": test_code})
review = json.loads(result.content)

print(f"Score: {review['quality_score']}/10")
print(f"Issues: {review['issues']}")
print(f"Suggestions: {review['suggestions']}")
```

</details>

---

## 💡 Interview Prep: Common Questions

```
╔═══════════════════════════════════════════════════════════════════════╗
║           INTERVIEW QUESTIONS FOR THIS MODULE                        ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  Q: What is few-shot prompting and when would you use it?            ║
║  A: Providing examples in the prompt to guide the model's output    ║
║     format and behavior. Use when: you need consistent output       ║
║     format, the task is nuanced, or zero-shot gives inconsistent    ║
║     results. Trade-off: uses more tokens (examples cost money).     ║
║                                                                       ║
║  Q: How do you get reliable JSON output from an LLM?                ║
║  A: 1) Use a clear system prompt specifying the exact JSON schema   ║
║     2) Say "Return ONLY valid JSON, no other text"                  ║
║     3) Use temperature=0 for consistency                             ║
║     4) In production: use Pydantic models with structured output    ║
║        (response_format) or output parsers                          ║
║     5) Always wrap in try/except for JSON parse failures            ║
║                                                                       ║
║  Q: What's the difference between a system and user message?        ║
║  A: System message sets behavior/personality (invisible to user).   ║
║     User message is the actual query. System persists across        ║
║     the conversation. Think of system as "background instructions"  ║
║     and user as "foreground requests".                               ║
║                                                                       ║
║  Q: What is a prompt template? Why use one?                          ║
║  A: A reusable prompt with variables (placeholders). Benefits:      ║
║     version control, consistency, composability with chains,        ║
║     separation of prompt logic from application logic.              ║
║     LangChain's ChatPromptTemplate is the standard.                 ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Quick Quiz

---

### Question 1:
You need the LLM to classify emails into exactly 5 categories with a specific JSON format. Which approach is best?

- A) Zero-shot with vague instructions
- B) Few-shot with examples showing each category
- C) Just ask nicely
- D) Use temperature=2.0

<details><summary>Answer</summary>

**B) Few-shot with examples showing each category**

When you need a specific output format with specific categories, few-shot examples are the most reliable approach. Show 1-2 examples per category so the model learns the exact format and classification criteria.

</details>

---

### Question 2:
What does the `|` operator do in LangChain's LCEL?

- A) Logical OR operation
- B) String concatenation
- C) Chains components together (output of left → input of right)
- D) Parallel execution

<details><summary>Answer</summary>

**C) Chains components together (output of left → input of right)**

`prompt | model` means: format the prompt → pass to model → get response. You can chain any number of components: `prompt | model | parser | validator`. This is LCEL — LangChain Expression Language.

</details>

---

### Question 3:
Your LLM returns JSON but sometimes adds extra text like "Here is the JSON:". How do you fix this?

- A) Hope it stops
- B) Use temperature=0 and add "Return ONLY valid JSON, no other text" to the system prompt
- C) Parse the response with regex
- D) Use a different model

<details><summary>Answer</summary>

**B) Use temperature=0 and add "Return ONLY valid JSON, no other text" to the system prompt**

This is the most reliable fix. Additionally, in production you'd use `response_format={"type": "json_object"}` in the API call (OpenAI's JSON mode) or Pydantic output parsers in LangChain to guarantee valid JSON.

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 Zero-shot = no examples. Few-shot = examples guide format.    │
├─────────────────────────────────────────────────────────────────────┤
│  📌 System prompt = role/persona. Completely changes behavior.     │
├─────────────────────────────────────────────────────────────────────┤
│  📌 ChatPromptTemplate = reusable prompt with {variables}         │
├─────────────────────────────────────────────────────────────────────┤
│  📌 LCEL: prompt | model = chain. The core pattern in LangChain.  │
├─────────────────────────────────────────────────────────────────────┤
│  📌 For JSON: specify schema + temp=0 + "ONLY valid JSON"         │
├─────────────────────────────────────────────────────────────────────┤
│  📌 In production: Pydantic models + output parsers for safety    │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Advanced Prompting Techniques!

Chain-of-Thought, Tree-of-Thought, ReAct, and structured outputs with Pydantic. The prompting gets serious!

---

### 🎉 Excellent Work!
You've mastered the core prompting patterns and built a real product review analyzer. You're thinking like a prompt engineer! 🚀